# CP1 — EDA и подготовка данных

**Задача:** Регрессия — предсказание концентрации PM2.5 (мкг/м³) в атмосфере Пекина.  
**Датасет:** Beijing Multi-Site Air-Quality Data (UCI / Kaggle)  
**Источник:** https://www.kaggle.com/datasets/sid321axn/beijing-multisite-airquality-data-set  

**Почему выбран датасет:**  
- Более 420 000 строк, 12 станций мониторинга, период 2013–2017.  
- Реальная экологическая задача с практическим значением.  
- Богатый набор фич: химические показатели + метеорология + временные паттерны.

**Метрика качества:** RMSE (Root Mean Squared Error)  
- Штрафует большие ошибки сильнее, что важно для задачи: ошибочное предсказание чистого воздуха при высоком PM2.5 опасно для здоровья.  
- Дополнительно смотрим MAE (интерпретируемость) и R² (доля объяснённой дисперсии).

**Структура ноутбука:**  
1. Загрузка и первичный осмотр данных  
2. Анализ пропусков  
3. Анализ выбросов  
4. Распределения признаков  
5. Корреляционный анализ  
6. Временные паттерны  
7. Обработка данных  
8. Feature Engineering  
9. Сплит train / val / test  
10. Сохранение обработанных данных

In [ ]:
import sys
import os

# Добавляем корень проекта в путь, чтобы импортировать src
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    load_raw_data, make_datetime, remove_duplicates,
    handle_missing_values, remove_outliers_iqr,
    encode_wind_direction, encode_station,
    add_time_features, add_lag_features, add_rolling_features,
    time_based_split, WIND_DIR_MAP,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 1. Загрузка данных

In [ ]:
df_raw = load_raw_data(RAW_DIR)
print(f"Строк: {len(df_raw):,}  |  Колонок: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
print("Типы данных:")
print(df_raw.dtypes)
print("\nСтанции:", df_raw["station"].unique())
print(f"Период: {df_raw['year'].min()} – {df_raw['year'].max()}")

In [ ]:
df_raw.describe()

## 2. Анализ пропусков

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["count"] > 0].sort_values("pct", ascending=False)
print(missing_df)

fig, ax = plt.subplots(figsize=(10, 4))
missing_df["pct"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Доля пропусков по колонкам (%)", fontsize=13)
ax.set_ylabel("%")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "missing_values.png"), dpi=100)
plt.show()

**Вывод:** PM2.5 и другие концентрации загрязнителей имеют пропуски ~3-5%. Стратегия — forward-fill по станции (данные временные, значение предыдущего часа близко к текущему), затем median fill для оставшихся.

## 3. Анализ распределения целевой переменной

In [ ]:
os.makedirs(os.path.join(PROJECT_ROOT, "report", "images"), exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_raw["PM2.5"].dropna().plot(
    kind="hist", bins=80, ax=axes[0], color="steelblue", edgecolor="white"
)
axes[0].set_title("Распределение PM2.5")
axes[0].set_xlabel("мкг/м³")

np.log1p(df_raw["PM2.5"].dropna()).plot(
    kind="hist", bins=80, ax=axes[1], color="coral", edgecolor="white"
)
axes[1].set_title("log1p(PM2.5) — ближе к нормальному")
axes[1].set_xlabel("log1p(мкг/м³)")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "pm25_distribution.png"), dpi=100)
plt.show()

print(f"PM2.5 — медиана: {df_raw['PM2.5'].median():.1f}, среднее: {df_raw['PM2.5'].mean():.1f}, макс: {df_raw['PM2.5'].max():.1f}")

**Вывод:** Распределение правостороннее (long tail). Максимальные значения PM2.5 > 700 мкг/м³ — явные выбросы (норма ВОЗ — 25 мкг/м³ в сутки). Будем удалять экстремальные выбросы по методу 3*IQR.

## 4. Анализ выбросов по всем числовым признакам

In [ ]:
num_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df_raw[col].dropna().plot(kind="box", ax=axes[i], color="steelblue")
    axes[i].set_title(col, fontsize=10)

# Скрываем пустой subplot
axes[-1].set_visible(False)
plt.suptitle("Boxplot по числовым признакам", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "boxplots.png"), dpi=100)
plt.show()

## 5. Корреляционный анализ

In [ ]:
corr_df = df_raw[num_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, ax=ax, square=True, linewidths=0.5
)
ax.set_title("Матрица корреляций (числовые признаки)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "correlation_heatmap.png"), dpi=100)
plt.show()

print("Корреляция с PM2.5:")
print(corr_matrix["PM2.5"].sort_values(ascending=False))

## 6. Временные паттерны

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# По часам суток
hourly = df_raw.groupby("hour")["PM2.5"].median()
hourly.plot(ax=axes[0], color="steelblue", marker="o", markersize=4)
axes[0].set_title("PM2.5 по часам суток (медиана)")
axes[0].set_xlabel("Час")
axes[0].set_ylabel("мкг/м³")

# По месяцам
monthly = df_raw.groupby("month")["PM2.5"].median()
monthly.plot(ax=axes[1], color="coral", marker="o", markersize=4)
axes[1].set_title("PM2.5 по месяцам (медиана)")
axes[1].set_xlabel("Месяц")

# По годам
yearly = df_raw.groupby("year")["PM2.5"].median()
yearly.plot(ax=axes[2], color="green", marker="o", markersize=4)
axes[2].set_title("PM2.5 по годам (медиана)")
axes[2].set_xlabel("Год")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "time_patterns.png"), dpi=100)
plt.show()

In [ ]:
# PM2.5 по станциям
fig, ax = plt.subplots(figsize=(12, 5))
station_median = df_raw.groupby("station")["PM2.5"].median().sort_values(ascending=False)
station_median.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Медианный PM2.5 по станциям")
ax.set_ylabel("мкг/м³")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "pm25_by_station.png"), dpi=100)
plt.show()

In [ ]:
# Временной ряд — агрегированный по дням (одна станция для примера)
station_ex = df_raw["station"].value_counts().index[0]
ts = df_raw[df_raw["station"] == station_ex].copy()
ts = make_datetime(ts).set_index("datetime")["PM2.5"].resample("D").median()

fig, ax = plt.subplots(figsize=(14, 4))
ts.plot(ax=ax, alpha=0.7, color="steelblue")
ax.set_title(f"Временной ряд PM2.5 (станция {station_ex}, дневные медианы)")
ax.set_ylabel("мкг/м³")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "timeseries.png"), dpi=100)
plt.show()

## 7. Обработка данных

In [ ]:
df = load_raw_data(RAW_DIR)

# 1. Создаём datetime
df = make_datetime(df)
print(f"После make_datetime: {len(df):,} строк")

# 2. Дубликаты
df = remove_duplicates(df)
print(f"После remove_duplicates: {len(df):,} строк")

# 3. Пропуски
df = handle_missing_values(df)
print(f"После handle_missing_values: {len(df):,} строк")
print(f"Оставшиеся пропуски: {df.isnull().sum().sum()}")

# 4. Выбросы
df = remove_outliers_iqr(df, target_col="PM2.5")
print(f"После remove_outliers_iqr: {len(df):,} строк")

## 8. Feature Engineering

Признаки, которые добавляем:

| Признак | Описание | Мотивация |
|---|---|---|
| `hour_sin`, `hour_cos` | Циклическое кодирование часа | Час 23 и 0 близки |
| `month_sin`, `month_cos` | Циклическое кодирование месяца | Сезонность |
| `day_of_week`, `is_weekend` | День недели | Трафик / промышленность |
| `wd_sin`, `wd_cos` | Направление ветра как вектор | Непрерывное кодирование угла |
| `station_enc` | Числовой код станции | Разные районы города |
| `PM2.5_lag1/2/3/24` | Лаги цели | Авторегрессия: воздух инерционен |
| `PM2.5_roll3`, `PM2.5_roll24` | Скользящее среднее | Тренд за последние 3/24 часа |

In [ ]:
df = encode_wind_direction(df)
df = encode_station(df)
df = add_time_features(df)
df = add_lag_features(df, target_col="PM2.5", lags=[1, 2, 3, 24])
df = add_rolling_features(df, target_col="PM2.5", windows=[3, 24])

print(f"Итоговое число признаков: {df.shape[1]}")
print(f"Строк: {len(df):,}")
df.head(3)

In [ ]:
# Scatter plot lag1 vs PM2.5
sample = df.sample(5000, random_state=RANDOM_SEED)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sample["PM2.5_lag1"], sample["PM2.5"], alpha=0.2, s=5, color="steelblue")
ax.set_xlabel("PM2.5 (t-1)")
ax.set_ylabel("PM2.5 (t)")
ax.set_title("PM2.5 vs Lag1")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "lag1_scatter.png"), dpi=100)
plt.show()

## 9. Train / Val / Test сплит

**Стратегия:** сплит по времени, а не случайный.  
**Почему:** данные временные (time series). Случайный сплит привёл бы к **data leakage** — модель "видела" бы будущие значения при обучении (лаги и скользящие средние), что нереалистично.  

Разбиение:
- **Train:** 2013–2015 (3 года, большинство данных)
- **Val:** 2016 (для подбора гиперпараметров)
- **Test:** 2017 (финальная оценка, не трогаем до конца)

In [ ]:
train, val, test = time_based_split(df, val_year=2016, test_year=2017)

print(f"Train: {len(train):,} строк  ({train['year'].min()}–{train['year'].max()})")
print(f"Val:   {len(val):,} строк  ({val['year'].min()}–{val['year'].max()})")
print(f"Test:  {len(test):,} строк  ({test['year'].min()}–{test['year'].max()})")

total = len(train) + len(val) + len(test)
print(f"\nДоли: train {len(train)/total:.1%} / val {len(val)/total:.1%} / test {len(test)/total:.1%}")

In [ ]:
# Проверка: нет пересечений по датам
assert train["datetime"].max() < val["datetime"].min(), "Leak: train/val пересекаются!"
assert val["datetime"].max() < test["datetime"].min(), "Leak: val/test пересекаются!"
print("Проверка на data leakage: OK")

## 10. Сохранение обработанных данных

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

train.to_parquet(os.path.join(PROCESSED_DIR, "train.parquet"), index=False)
val.to_parquet(os.path.join(PROCESSED_DIR, "val.parquet"), index=False)
test.to_parquet(os.path.join(PROCESSED_DIR, "test.parquet"), index=False)

print("Сохранено:")
for split, ds in [("train", train), ("val", val), ("test", test)]:
    path = os.path.join(PROCESSED_DIR, f"{split}.parquet")
    size_mb = os.path.getsize(path) / 1024**2
    print(f"  {split}.parquet — {len(ds):,} строк, {size_mb:.1f} MB")

## Итоги EDA

- **Пропуски:** ~3-5% в химических показателях, заполнены forward-fill по станции + median fallback.  
- **Выбросы:** удалены экстремальные значения PM2.5 (> Q3 + 3*IQR) — около 1% данных.  
- **Самые коррелированные с PM2.5:** PM10, CO, NO2 (r > 0.7). O3 — отрицательная корреляция (фотохимическая реакция).  
- **Временные паттерны:** пик PM2.5 зимой (отопление), пиковые часы — утро и вечер (трафик).  
- **Feature engineering:** добавлены лаги 1/2/3/24 часа и скользящие средние 3/24 часа — PM2.5 сильно авторегрессивный.  
- **Сплит:** по времени, train 2013–2015 / val 2016 / test 2017. Data leakage исключён.